In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
import os, sys, importlib
import mlflow

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.append("/Workspace" + os.path.dirname(nb_path))
importlib.invalidate_caches()

from mlflow.deployments import get_deploy_client
from retrieval import retrieve_context

deploy = get_deploy_client("databricks")
LLM = "databricks-meta-llama-3-3-70b-instruct"

SYSTEM_PROMPT = """You are PrivacyLens, an educational assistant for India's DPDP Act 2023 and the EU GDPR.
Rules:
1. Answer ONLY using the context provided. Do not use outside knowledge.
2. Cite every claim with its source in square brackets, exactly as labelled in the context, e.g. [GDPR Article 17] or [DPDP Section 12].
3. If the context does not contain the answer, reply exactly: "I could not find this in the GDPR or DPDP Act text provided."
4. When both laws are relevant, explain each and note key differences.
5. Keep the answer under 200 words. This is not legal advice."""

def short_cite(h):
    if h["section_num"] == "Schedule":
        return f"{h['law']} Schedule"
    unit = "Article" if h["law"] == "GDPR" else "Section"
    return f"{h['law']} {unit} {h['section_num']}"

def build_context(hits):
    return "\n\n---\n\n".join(f"[{short_cite(h)}]\n{h['chunk_text']}" for h in hits)

@mlflow.trace(span_type="RETRIEVER")
def traced_retrieve(question, law=None):
    return retrieve_context(question, law)

@mlflow.trace(span_type="LLM")
def call_llm(messages):
    resp = deploy.predict(endpoint=LLM, inputs={
        "messages": messages, "max_tokens": 600, "temperature": 0,
    })
    return resp["choices"][0]["message"]["content"]

@mlflow.trace(span_type="CHAIN")
def answer(question, law=None):
    hits = traced_retrieve(question, law)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{build_context(hits)}\n\nQuestion: {question}"},
    ]
    return call_llm(messages), hits

In [0]:
for q in [
    "Can a user make a company delete their data?",
    "What is the capital gains tax rate in India?",
]:
    text, hits = answer(q)
    print("Q:", q)
    print("Sources retrieved:", sorted({short_cite(h) for h in hits}))
    print("A:", text)
    print("=" * 80)

In [0]:
import os, mlflow, pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from mlflow.models.resources import (
    DatabricksVectorSearchIndex, DatabricksServingEndpoint,
)

folder = "/Workspace" + os.path.dirname(nb_path)

signature = ModelSignature(
    inputs=Schema([
        ColSpec("string", "question"),
        ColSpec("string", "law", required=False),
    ]),
    outputs=Schema([
        ColSpec("string", "answer"),
        ColSpec("string", "sources"),
    ]),
)

with mlflow.start_run(run_name="privacylens_v1"):
    info = mlflow.pyfunc.log_model(
        name="privacylens",
        python_model=f"{folder}/privacylens_model.py",
        code_paths=[f"{folder}/retrieval.py"],
        signature=signature,
        input_example=pd.DataFrame({"question": ["Can a user make a company delete their data?"]}),
        pip_requirements=["mlflow", "databricks-ai-search", "pandas"],
        resources=[
            DatabricksVectorSearchIndex(index_name="workspace.privacylens.silver_chunks_index"),
            DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-3-70b-instruct"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
        ],
    )

print("Logged:", info.model_uri)

In [0]:
loaded = mlflow.pyfunc.load_model(info.model_uri)
out = loaded.predict(pd.DataFrame({"question": ["When must a personal data breach be reported?"]}))
print("Sources:", out.loc[0, "sources"])
print("Answer:", out.loc[0, "answer"])

In [0]:
mlflow.models.predict(
    model_uri=info.model_uri,
    input_data=pd.DataFrame({"question": ["When must a personal data breach be reported?"]}),
    env_manager="uv",
)